# Phase 2 — Conjugate Families

Companion notebook to `notes/phase2-conjugate-families.md`. We exercise
all four conjugate updaters end-to-end against the FYF reference
parameters, cross-check every numerical example from the theory notes
and exercises, and visualise:

1. Normal-Normal update for Q1 actuals.
2. Posterior trajectory across n = 1, 3, 6, 12.
3. Gamma-Poisson update for the incident-rate prior.
4. Beta-Binomial update for the overtime-proportion prior.
5. Normal-Inverse-Gamma update with three Q1 actuals (unknown σ²).

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from src.conjugate import (
    NormalNormalUpdater,
    GammaPoissonUpdater,
    BetaBinomialUpdater,
    NormalInverseGammaUpdater,
)

rng = np.random.default_rng(seed=20260512)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## 1. Normal-Normal update for Q1 actuals

Reference parameters: $\mu_0 = 1{,}050{,}000$, $\sigma_0 = 150{,}000$,
$\sigma = 80{,}000$. Q1 actuals from `docs/model-design.md` §5.

In [ ]:
mu0, sigma0, sigma = 1_050_000.0, 150_000.0, 80_000.0
actuals = [1_120_000.0, 1_080_000.0, 1_095_000.0]

upd = NormalNormalUpdater(mu0=mu0, sigma0_sq=sigma0**2, sigma_sq=sigma**2)
post = upd.update(actuals)
s = post.summary(level=0.95)
lo, hi = s["credible_interval"]
print(f"posterior mean μ_3   = R$ {s['mean']:>13,.0f}")
print(f"posterior s.d. σ_3   = R$ {s['std']:>13,.0f}")
print(f"95% credible CI       = [R$ {lo:,.0f}, R$ {hi:,.0f}]")
print(f"posterior precision   = {s['precision']:.3e}")

Visual: prior, scaled likelihood kernel, and posterior on the same axis.
(The standalone figure used in the article comes from
`scripts/fig_normal_normal_update.py`.)

In [ ]:
x_bar = float(np.mean(actuals))
n = len(actuals)
grid = np.linspace(mu0 - 4*sigma0, mu0 + 4*sigma0, 800)

prior_pdf = stats.norm(mu0, sigma0).pdf(grid)
lik_kernel = stats.norm(x_bar, sigma/np.sqrt(n)).pdf(grid)
post_pdf = stats.norm(post.mean(), post.std()).pdf(grid)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(grid, prior_pdf, color="steelblue", lw=2, label=f"Prior  N({mu0:,.0f}, {sigma0:,.0f}²)")
ax.plot(grid, lik_kernel, color="goldenrod", lw=2, ls="--",
        label=f"Likelihood kernel  N({x_bar:,.0f}, σ²/n)")
ax.plot(grid, post_pdf, color="crimson", lw=2,
        label=f"Posterior  N({post.mean():,.0f}, {post.std():,.0f}²)")
for x, c, lbl in [(mu0, "steelblue", "μ₀"), (x_bar, "goldenrod", "x̄"), (post.mean(), "crimson", "μₙ")]:
    ax.axvline(x, color=c, ls=":", alpha=0.5)
ax.set_xlabel(r"θ — mean monthly cost (R$)")
ax.set_ylabel("density")
ax.legend(fontsize=9, loc="upper left")
ax.ticklabel_format(style="plain", axis="x")
fig.tight_layout()
plt.show()

## 2. Posterior trajectory across 1, 3, 6, 12 months (Exercise 7)

Using fabricated sample means at each revision point to illustrate
shrinkage of σₙ and the trajectory of μₙ.

In [ ]:
rows = [(1, 1_120_000), (3, 1_095_000), (6, 1_085_000), (12, 1_078_000)]
print(f"{'n':>3} | {'x_bar':>14} | {'mu_n':>14} | {'sigma_n':>10} | 95% CI")
print("-" * 80)
for n, xbar in rows:
    p = upd.update(np.full(n, float(xbar)))
    s = p.summary(level=0.95)
    lo, hi = s['credible_interval']
    print(f"{n:>3} | R$ {xbar:>10,.0f} | R$ {s['mean']:>10,.0f} | R$ {s['std']:>7,.0f} | [R$ {lo:,.0f}, R$ {hi:,.0f}]")

In [ ]:
# Forest plot: 95% CI at each revision
fig, ax = plt.subplots(figsize=(8, 3.6))
for i, (n, xbar) in enumerate(rows):
    p = upd.update(np.full(n, float(xbar)))
    lo, hi = p.credible_interval(0.95)
    ax.errorbar(p.mean(), i, xerr=[[p.mean()-lo],[hi-p.mean()]], fmt="o",
                color="crimson", capsize=4)
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([f"n={n}\nx̄={xbar:,.0f}" for n, xbar in rows])
ax.invert_yaxis()
ax.axvline(mu0, color="steelblue", ls=":", alpha=0.6, label=f"prior mean μ₀={mu0:,.0f}")
ax.set_xlabel(r"posterior mean μₙ and 95% credible interval (R$)")
ax.set_title("Shrinkage of the posterior across the FYF cycle")
ax.ticklabel_format(style="plain", axis="x")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 3. Gamma-Poisson — incident-rate update (Exercise 8)

Prior $\lambda \sim \text{Gamma}(3, 1)$. Six observed monthly incident
counts: $\{2, 4, 1, 3, 5, 2\}$.

In [ ]:
data_inc = np.array([2, 4, 1, 3, 5, 2])
gp_upd = GammaPoissonUpdater(alpha0=3.0, beta0=1.0)
gp_post = gp_upd.update(data_inc)
print(f"posterior  Gamma({gp_post.alpha:.1f}, {gp_post.beta:.1f})")
print(f"posterior mean = {gp_post.mean():.4f}")
print(f"sample mean    = {data_inc.mean():.4f}")
print(f"prior mean     = 3.0000")
lo, hi = gp_post.credible_interval(0.95)
print(f"95% CI         = [{lo:.3f}, {hi:.3f}]")

# Compare with a stronger prior — Gamma(30, 10), same prior mean.
gp_strong = GammaPoissonUpdater(alpha0=30.0, beta0=10.0).update(data_inc)
print(f"\nstronger prior posterior  Gamma({gp_strong.alpha:.1f}, {gp_strong.beta:.1f}), mean={gp_strong.mean():.4f}")
print("(closer to the prior mean of 3 because the prior carries 10 months of pseudo-data)")

In [ ]:
# Visualise both posteriors against the prior
lam_grid = np.linspace(0, 8, 400)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(lam_grid, stats.gamma(3.0, scale=1.0).pdf(lam_grid),
        label="Prior  Gamma(3, 1)", color="steelblue")
ax.plot(lam_grid, stats.gamma(gp_post.alpha, scale=1/gp_post.beta).pdf(lam_grid),
        label=f"Weak prior posterior  Gamma({gp_post.alpha:.0f}, {gp_post.beta:.0f})",
        color="crimson")
ax.plot(lam_grid, stats.gamma(gp_strong.alpha, scale=1/gp_strong.beta).pdf(lam_grid),
        label=f"Strong prior posterior  Gamma({gp_strong.alpha:.0f}, {gp_strong.beta:.0f})",
        color="darkgreen", ls="--")
ax.axvline(data_inc.mean(), color="black", ls=":", alpha=0.6, label=f"sample mean = {data_inc.mean():.3f}")
ax.set_xlabel("λ — incidents per month")
ax.set_ylabel("density")
ax.legend(fontsize=9)
ax.set_title("Gamma-Poisson: prior strength governs how much data overrides the prior")
fig.tight_layout()
plt.show()

## 4. Beta-Binomial — overtime-proportion update

Default prior $\text{Beta}(2, 8)$ (mean 0.20, weakly informative). We
simulate observing 12 months of overtime data: 50 trials per month with
12 "successes" (i.e. 24 % observed).

In [ ]:
bb_upd = BetaBinomialUpdater(alpha0=2.0, beta0=8.0)
bb_post = bb_upd.update(successes=12, trials=50)
print(f"posterior Beta({bb_post.alpha:.0f}, {bb_post.beta:.0f}), mean = {bb_post.mean():.4f}")
lo, hi = bb_post.credible_interval(0.95)
print(f"95% credible CI = [{lo:.3f}, {hi:.3f}]")

## 5. Normal-Inverse-Gamma — unknown σ²

Prior chosen so that $\mathbb E[\sigma^2] = 80{,}000^2 = 6.4 \times 10^9$
(matching the known-σ assumption used elsewhere). Three Q1 actuals.

In [ ]:
nig_upd = NormalInverseGammaUpdater(
    mu0=1_050_000.0, kappa0=1.0, alpha0=2.0, beta0=6.4e9
)
nig_post = nig_upd.update(actuals)
print(f"μ_n     = {nig_post.mu:,.0f}")
print(f"κ_n     = {nig_post.kappa}")
print(f"α_n     = {nig_post.alpha}")
print(f"β_n     = {nig_post.beta:.3e}")
print(f"E[σ²]   = {nig_post.mean_sigma_sq():.3e}  (i.e. σ ≈ R$ {np.sqrt(nig_post.mean_sigma_sq()):,.0f})")
lo, hi = nig_post.credible_interval(0.95)
print(f"95% CI  = [{lo:,.0f}, {hi:,.0f}] (Student-t marginal)")

Comparing with the Normal-Normal posterior (which assumed σ = 80K
known), the NIG credible interval should be wider because we now also
absorb uncertainty about σ². The amount of widening is the **cost of
admitting that σ² is also unknown** — small here because $\alpha_0 = 2$
puts moderate confidence on σ².

---

**Next phase.** `notes/phase3-sequential-updating.md` proves that
feeding data sequentially (one month at a time) yields the same
posterior as feeding all data in one batch — so the FYF cycle
literally is a chain of conjugate updates. Then we derive the
shrinkage rate and analyse prior sensitivity.